# Shield of Mercy Energy Dome master specification tables for architecture, design targets, interfaces, risks, and validation planning Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.hypc-w238/fair2.json`

This notebook follows a reusable exploration pattern:
- load Croissant metadata,
- inspect record sets and fields using their `@id` values,
- load records into pandas DataFrames,
- perform light exploratory data analysis,
- and generate simple visual summaries.

> **Important:** In Croissant datasets, record sets, fields, and other entities should be referenced by their `@id` values. This notebook dynamically discovers those identifiers from the metadata wherever possible.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`, then inspect the dataset-level description.

In [ ]:
import json
import math
import warnings

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.hypc-w238/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON-like structure for display
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print("Dataset name:", metadata_json.get("name", "N/A"))
print("Version:", metadata_json.get("version", "N/A"))
print("Identifier:", metadata_json.get("identifier", "N/A"))
print("License:", metadata_json.get("license", "N/A"))
print("Description:\n", metadata_json.get("description", "N/A"))

In [ ]:
# Dataset-level summary fields
summary = {
    "name": metadata_json.get("name"),
    "datePublished": metadata_json.get("datePublished"),
    "temporalCoverage": metadata_json.get("temporalCoverage"),
    "version": metadata_json.get("version"),
    "keywords_count": len(metadata_json.get("keywords", [])) if isinstance(metadata_json.get("keywords", []), list) else 0,
    "distribution_count": len(metadata_json.get("distribution", [])) if isinstance(metadata_json.get("distribution", []), list) else 0,
    "recordSet_count": len(metadata_json.get("recordSet", [])) if isinstance(metadata_json.get("recordSet", []), list) else 0,
}

pd.DataFrame([summary])

## 2. Data Overview
Review the available record sets, fields, and their `@id` values.

The provided FAIR^2 package summary indicates that the top-level `recordSet` list is empty. In practice, some Croissant sources expose structure differently depending on the endpoint and serialization.

The following cells:
1. inspect the raw JSON metadata,
2. extract any visible record sets and fields,
3. and attempt to preview records if record sets are available.

In [ ]:
# Inspect top-level metadata keys
sorted(metadata_json.keys())

In [ ]:
# Extract record sets from the JSON representation, if present
record_sets_json = metadata_json.get("recordSet", [])

print(f"Number of record sets discovered in metadata JSON: {len(record_sets_json)}")

record_set_rows = []
for rs in record_sets_json:
    record_set_rows.append({
        "record_set_id": rs.get("@id"),
        "name": rs.get("name"),
        "description": rs.get("description"),
        "field_count": len(rs.get("field", [])) if isinstance(rs.get("field", []), list) else 0,
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df if not record_sets_df.empty else pd.DataFrame({"message": ["No record sets were listed in the top-level metadata JSON."]})

In [ ]:
# Extract field-level details, always using @id when available
field_rows = []

for rs in record_sets_json:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    fields = rs.get("field", []) if isinstance(rs.get("field", []), list) else []
    for fld in fields:
        source_info = fld.get("source", {}) if isinstance(fld.get("source", {}), dict) else {}
        field_rows.append({
            "record_set_id": rs_id,
            "record_set_name": rs_name,
            "field_id": fld.get("@id"),
            "field_name": fld.get("name"),
            "data_type": fld.get("dataType"),
            "description": fld.get("description"),
            "source_file_object_id": source_info.get("fileObject"),
            "source_extract_column": source_info.get("extract", {}).get("column") if isinstance(source_info.get("extract", {}), dict) else None,
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(20) if not fields_df.empty else pd.DataFrame({"message": ["No field definitions were found under record sets in the top-level metadata JSON."]})

### Record preview
If record sets are available, we can inspect a small sample of records from a selected record set by its `@id`.

In [ ]:
# Build a reusable list of record set @id values
record_set_ids = record_sets_df["record_set_id"].dropna().tolist() if not record_sets_df.empty else []
print("Record set @id values discovered:")
print(record_set_ids if record_set_ids else "None discovered from metadata JSON")

preview_record_set_id = record_set_ids[0] if record_set_ids else None

if preview_record_set_id is not None:
    print(f"Previewing up to 3 records from record set: {preview_record_set_id}")
    sample_records = []
    for i, rec in enumerate(dataset.records(record_set=preview_record_set_id)):
        sample_records.append(rec)
        if i >= 2:
            break
    for rec in sample_records:
        print(rec)
else:
    print("No record set @id was available to preview records.")

## 3. Data Extraction
Load data from discovered record sets into pandas DataFrames.

This section is fully dynamic:
- it uses the discovered record set `@id` values,
- stores each loaded table in a dictionary keyed by record set `@id`,
- and reports shape and columns for each loaded DataFrame.

In [ ]:
dataframes = {}
load_log = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        load_log.append({
            "record_set_id": record_set_id,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "status": "loaded",
        })
    except Exception as e:
        load_log.append({
            "record_set_id": record_set_id,
            "rows": None,
            "columns": None,
            "status": f"failed: {type(e).__name__}: {e}",
        })

load_log_df = pd.DataFrame(load_log)
load_log_df if not load_log_df.empty else pd.DataFrame({"message": ["No record sets were available to load."]})

In [ ]:
# Inspect the first successfully loaded DataFrame, if any
loaded_record_set_ids = [rs_id for rs_id, df in dataframes.items() if isinstance(df, pd.DataFrame)]
selected_record_set_id = loaded_record_set_ids[0] if loaded_record_set_ids else None

print("Selected record set @id:", selected_record_set_id)

if selected_record_set_id is not None:
    print("Columns:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No DataFrame was successfully loaded from the discovered record sets.")

### Optional metadata-only fallback
If no record sets are exposed through the loaded Croissant endpoint, we can still explore the dataset package metadata itself as a structured object.

In [ ]:
# Metadata-only exploration tables
authors_df = pd.DataFrame(metadata_json.get("author", [])) if isinstance(metadata_json.get("author", []), list) else pd.DataFrame()
distributions_df = pd.DataFrame(metadata_json.get("distribution", [])) if isinstance(metadata_json.get("distribution", []), list) else pd.DataFrame()
keywords_df = pd.DataFrame({"keyword": metadata_json.get("keywords", [])}) if isinstance(metadata_json.get("keywords", []), list) else pd.DataFrame()
biases_df = pd.DataFrame({"dataBias": metadata_json.get("dataBiases", [])}) if isinstance(metadata_json.get("dataBiases", []), list) else pd.DataFrame()
limitations_df = pd.DataFrame({"dataLimitation": metadata_json.get("dataLimitations", [])}) if isinstance(metadata_json.get("dataLimitations", []), list) else pd.DataFrame()

print("Authors")
display(authors_df.head())

print("Distributions")
display(distributions_df.head())

print("Keywords")
display(keywords_df.head(10))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to a loaded record set when available.

Because this dataset may expose multiple tables with different schemas, the notebook automatically:
1. selects a loaded record set,
2. detects numeric and categorical columns,
3. filters records on a numeric threshold,
4. normalizes one numeric field,
5. and groups by a categorical field where possible.

If no record set data are available, the notebook falls back to simple EDA on metadata tables such as keywords and distributions.

In [ ]:
def find_analysis_columns(df: pd.DataFrame):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()
    return numeric_cols, categorical_cols

if selected_record_set_id is not None and selected_record_set_id in dataframes:
    analysis_df = dataframes[selected_record_set_id].copy()
    numeric_cols, categorical_cols = find_analysis_columns(analysis_df)
    
    print("Numeric columns:", numeric_cols)
    print("Categorical columns:", categorical_cols[:20])
    
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        numeric_series = pd.to_numeric(analysis_df[numeric_field_id], errors="coerce")
        valid_numeric = numeric_series.dropna()
        
        if len(valid_numeric) > 0:
            threshold = valid_numeric.median()
            filtered_df = analysis_df[numeric_series > threshold].copy()
            
            print(f"Using numeric field: {numeric_field_id}")
            print(f"Threshold (median): {threshold}")
            print(f"Filtered rows: {len(filtered_df)} of {len(analysis_df)}")
            display(filtered_df.head())
            
            std_val = valid_numeric.std()
            if pd.notna(std_val) and std_val != 0:
                filtered_df[f"{numeric_field_id}_normalized"] = (
                    pd.to_numeric(filtered_df[numeric_field_id], errors="coerce") - valid_numeric.mean()
                ) / std_val
            else:
                filtered_df[f"{numeric_field_id}_normalized"] = np.nan
            
            print("Normalized numeric field preview:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
            
            group_field_id = None
            for col in categorical_cols:
                if col != numeric_field_id and analysis_df[col].nunique(dropna=True) > 1:
                    group_field_id = col
                    break
            
            if group_field_id is not None and not filtered_df.empty:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=True).reset_index()
                print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head(10))
            else:
                print("No suitable categorical grouping field was found.")
        else:
            print("A numeric column exists, but it does not contain usable numeric values after coercion.")
    else:
        print("No numeric columns found in the selected record set; skipping numeric EDA.")
else:
    print("No record-set DataFrame available. Running metadata-level EDA instead.")
    
    metadata_eda = pd.DataFrame([
        {
            "n_authors": len(metadata_json.get("author", [])) if isinstance(metadata_json.get("author", []), list) else 0,
            "n_keywords": len(metadata_json.get("keywords", [])) if isinstance(metadata_json.get("keywords", []), list) else 0,
            "n_distributions": len(metadata_json.get("distribution", [])) if isinstance(metadata_json.get("distribution", []), list) else 0,
            "n_biases": len(metadata_json.get("dataBiases", [])) if isinstance(metadata_json.get("dataBiases", []), list) else 0,
            "n_limitations": len(metadata_json.get("dataLimitations", [])) if isinstance(metadata_json.get("dataLimitations", []), list) else 0,
            "n_use_cases": len(metadata_json.get("dataUseCases", [])) if isinstance(metadata_json.get("dataUseCases", []), list) else 0,
        }
    ])
    display(metadata_eda)
    
    if not keywords_df.empty:
        keywords_df["keyword_length"] = keywords_df["keyword"].astype(str).str.len()
        threshold = keywords_df["keyword_length"].median()
        filtered_df = keywords_df[keywords_df["keyword_length"] > threshold].copy()
        filtered_df["keyword_length_normalized"] = (
            filtered_df["keyword_length"] - keywords_df["keyword_length"].mean()
        ) / (keywords_df["keyword_length"].std() if keywords_df["keyword_length"].std() != 0 else 1)
        print(f"Filtered keywords with length > {threshold}:")
        display(filtered_df.head(10))

## 5. Visualization
Visualize either loaded record-set data or, if unavailable, the dataset metadata.

In [ ]:
sns.set_theme(style="whitegrid")

if selected_record_set_id is not None and selected_record_set_id in dataframes:
    viz_df = dataframes[selected_record_set_id].copy()
    numeric_cols = viz_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = viz_df.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()
    
    if numeric_cols:
        plot_numeric_field_id = numeric_cols[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(pd.to_numeric(viz_df[plot_numeric_field_id], errors="coerce").dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {plot_numeric_field_id}\n(record set: {selected_record_set_id})")
        plt.xlabel(plot_numeric_field_id)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric columns available for histogram in the selected record set.")
    
    if categorical_cols:
        plot_group_field_id = categorical_cols[0]
        top_counts = viz_df[plot_group_field_id].astype(str).value_counts().head(10)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=top_counts.index, y=top_counts.values)
        plt.title(f"Top categories in {plot_group_field_id}\n(record set: {selected_record_set_id})")
        plt.xlabel(plot_group_field_id)
        plt.ylabel("Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("No categorical columns available for bar chart in the selected record set.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    counts = {
        "authors": len(metadata_json.get("author", [])) if isinstance(metadata_json.get("author", []), list) else 0,
        "keywords": len(metadata_json.get("keywords", [])) if isinstance(metadata_json.get("keywords", []), list) else 0,
        "distributions": len(metadata_json.get("distribution", [])) if isinstance(metadata_json.get("distribution", []), list) else 0,
        "biases": len(metadata_json.get("dataBiases", [])) if isinstance(metadata_json.get("dataBiases", []), list) else 0,
        "limitations": len(metadata_json.get("dataLimitations", [])) if isinstance(metadata_json.get("dataLimitations", []), list) else 0,
    }
    
    sns.barplot(x=list(counts.keys()), y=list(counts.values()), ax=axes[0])
    axes[0].set_title("Metadata component counts")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=45)
    
    if not keywords_df.empty:
        keyword_lengths = keywords_df["keyword"].astype(str).str.len()
        sns.histplot(keyword_lengths, bins=15, kde=True, ax=axes[1])
        axes[1].set_title("Distribution of keyword lengths")
        axes[1].set_xlabel("Keyword length")
    else:
        axes[1].text(0.5, 0.5, "No keywords available", ha="center", va="center")
        axes[1].set_title("Keyword length distribution")
        axes[1].set_axis_off()
    
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to explore a Croissant-described dataset from its schema URL.

### Key takeaways
- The dataset metadata was successfully loaded from the Croissant source.
- The notebook dynamically inspected record sets and fields using `@id`-based references.
- Where record sets were available, records were loaded into pandas DataFrames for analysis.
- Where the endpoint exposed mainly metadata, the notebook still supported useful metadata-level exploration and visualization.

### Recommended next steps
- Inspect any discovered record set `@id` values and choose domain-relevant tables for deeper analysis.
- Use field `@id` values when selecting columns for joins, filtering, and validation.
- Extend the EDA section with domain-specific checks such as status-tag distributions, threat-class summaries, validation-stage coverage, risk counts, or interface relationships.
- If the dataset provider later exposes record sets more explicitly, rerun this notebook without structural changes; the dynamic discovery logic should adapt automatically.